In [0]:
# Create widgets
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

# Retrieve widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

print(f"source_system     : {source_system}")
print(f"table_name        : {table_name}")
print(f"landing_timestamp : {landing_timestamp}")
print(f"source_file_format: {source_file_format}")

In [0]:
landing_path = f'/Volumes/dbx_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

if source_file_format == 'csv':
    landing_df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(landing_path)
else:
    landing_df = spark.read.format('parquet').load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df \
    .withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
updated_df.write.mode("append").format("delta").saveAsTable(f"dbx_caresync.bronze.{table_name}")